In [1]:
!git clone https://github.com/TNKAAG/deepfake-detection.git

Cloning into 'deepfake-detection'...
remote: Enumerating objects: 50, done.
remote: Counting objects: 100% (50/50), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 50 (delta 15), reused 43 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (50/50), 2.09 MiB | 8.27 MiB/s, done.
Resolving deltas: 100% (15/15), done.


In [2]:
import os
from google.colab import drive
os.chdir('deepfake-detection')

drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
import gdown

file_id = '1dJa_3_QlUSFRY2-o6zOaTKWJQ0OhNRFi'
url = f'https://drive.google.com/uc?id={file_id}'
output = 'dataset.zip'

gdown.download(url, output, quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=1dJa_3_QlUSFRY2-o6zOaTKWJQ0OhNRFi
From (redirected): https://drive.google.com/uc?id=1dJa_3_QlUSFRY2-o6zOaTKWJQ0OhNRFi&confirm=t&uuid=9f747a23-0cc3-4393-b2a7-aeae0d4451d2
To: /content/deepfake-detection/dataset.zip
100%|██████████| 2.18G/2.18G [00:38<00:00, 56.3MB/s]


'dataset.zip'

In [4]:
# Unzip the file you just downloaded
!unzip -q dataset.zip -d ./data

In [ ]:
# Cell 1: Imports
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import cv2
from scipy.fftpack import dct
from torchvision import datasets
from torch.utils.data import DataLoader

# Cell 2: DCT helper
def apply_2d_dct(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)  # img is BGR from cv2.imread
    dct_transformed = dct(dct(gray.T, norm='ortho').T, norm='ortho')
    epsilon = 1e-12
    dct_log = np.log(np.abs(dct_transformed) + epsilon)
    return dct_log.astype(np.float32)

# Cell 3: Dataset (single, correct definition)
class FrequencyDataset(datasets.ImageFolder):
    def __getitem__(self, index):
        path, target = self.samples[index]
        sample = cv2.imread(path)
        if sample is None:
            return torch.zeros((1, 128, 128)), target
        sample = cv2.resize(sample, (128, 128))
        freq_sample = apply_2d_dct(sample)
        return torch.from_numpy(freq_sample).unsqueeze(0), target

# Cell 4: Loaders
train_dir = './data/train'
val_dir   = './data/val'

train_dataset = FrequencyDataset(train_dir)
train_loader  = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)

val_dataset = FrequencyDataset(val_dir)
val_loader  = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

print(f"Classes: {train_dataset.classes}")
print(f"Train size: {len(train_dataset)}")

# Cell 5: *** THE MISSING MODEL ***
class FreqCNN(nn.Module):
    def __init__(self):
        super(FreqCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),  # 1 channel (grayscale DCT)
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),           # 64x64

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),           # 32x32

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),           # 16x16
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 16 * 16, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 2)             # 2 classes: fake / real
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# Cell 6: Training loop
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model     = FreqCNN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
criterion = nn.CrossEntropyLoss()
epochs    = 10

best_val_acc = 0.0  # track best model

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss    = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        if i % 10 == 0:
            print(f"Epoch [{epoch+1}/{epochs}], Step [{i}/{len(train_loader)}], Loss: {loss.item():.4f}")
    # Validation
    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            total   += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_acc = 100 * correct / total
    print(f"End of Epoch {epoch+1} | Val Accuracy: {val_acc:.2f}%")

    # ✅ Save checkpoint every epoch
    torch.save(model.state_dict(), f"model_epoch_{epoch+1}.pth")

    # ✅ Save best model only
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_model.pth")
        print("🔥 Best model saved!")

Classes: ['fake', 'real']
Train size: 73391
Epoch [1/10], Step [0/2294], Loss: 0.8141
Epoch [1/10], Step [10/2294], Loss: 0.4576
Epoch [1/10], Step [20/2294], Loss: 0.4576
Epoch [1/10], Step [30/2294], Loss: 0.3385
Epoch [1/10], Step [40/2294], Loss: 0.4671
Epoch [1/10], Step [50/2294], Loss: 0.6130
Epoch [1/10], Step [60/2294], Loss: 0.7230
Epoch [1/10], Step [70/2294], Loss: 0.5442
Epoch [1/10], Step [80/2294], Loss: 0.4647
Epoch [1/10], Step [90/2294], Loss: 0.4951
Epoch [1/10], Step [100/2294], Loss: 0.5324
Epoch [1/10], Step [110/2294], Loss: 0.5247
Epoch [1/10], Step [120/2294], Loss: 0.3951
Epoch [1/10], Step [130/2294], Loss: 0.4467
Epoch [1/10], Step [140/2294], Loss: 0.4897
Epoch [1/10], Step [150/2294], Loss: 0.3139
Epoch [1/10], Step [160/2294], Loss: 0.3685
Epoch [1/10], Step [170/2294], Loss: 0.4060
Epoch [1/10], Step [180/2294], Loss: 0.4524
Epoch [1/10], Step [190/2294], Loss: 0.3018
Epoch [1/10], Step [200/2294], Loss: 0.4290
Epoch [1/10], Step [210/2294], Loss: 0.5113